In [0]:
-- 1. Volume: total rows and rows per file
SELECT source_file, COUNT(*) AS rows
FROM workspace.shop.bronze_orders
GROUP BY source_file;

In [0]:
-- 2. Duplicates: order_id values that appear more than once
SELECT order_id, COUNT(*) AS cnt
FROM workspace.shop.bronze_orders
GROUP BY order_id
HAVING COUNT(*) > 1;

In [0]:
-- 3. NULL / empty values in key columns
SELECT
  COUNT_IF(order_id    IS NULL OR order_id = '')    AS null_order_id,
  COUNT_IF(customer_id IS NULL OR customer_id = '') AS null_customer_id,
  COUNT_IF(order_ts    IS NULL OR order_ts = '')    AS null_order_ts,
  COUNT_IF(unit_price  IS NULL OR unit_price = '')  AS null_unit_price
FROM workspace.shop.bronze_orders;

In [0]:
-- 4. Types: values that cannot be cast to a number / date (everything is string in bronze)
SELECT
  COUNT_IF(TRY_CAST(quantity AS INT) IS NULL)          AS bad_quantity,
  COUNT_IF(TRY_CAST(unit_price AS DOUBLE) IS NULL)     AS bad_price,
  COUNT_IF(TRY_TO_TIMESTAMP(order_ts) IS NULL)         AS bad_order_ts
FROM workspace.shop.bronze_orders;

In [0]:
-- 5. Business logic: impossible values
SELECT
  COUNT_IF(CAST(quantity AS INT) <= 0)       AS qty_le_0,
  COUNT_IF(CAST(unit_price AS DOUBLE) <= 0)  AS price_le_0
FROM workspace.shop.bronze_orders;

In [0]:
-- 6. Reference values: check for typos and unexpected categories
SELECT 'currency' AS col, currency AS value, COUNT(*) AS cnt FROM workspace.shop.bronze_orders GROUP BY currency
UNION ALL
SELECT 'status', status, COUNT(*) FROM workspace.shop.bronze_orders GROUP BY status
UNION ALL
SELECT 'country', country, COUNT(*) FROM workspace.shop.bronze_orders GROUP BY country;

In [0]:
-- 7. Date range: check for orders from the future or from the year 1900
SELECT MIN(order_ts) AS min_ts, MAX(order_ts) AS max_ts
FROM workspace.shop.bronze_orders;

In [0]:
--CREATE OR REFRESH MATERIALIZED VIEW dq_bronze_orders AS
SELECT
  current_timestamp()                                   AS checked_at,        -- when the check ran
  COUNT(*)                                              AS total_rows,        -- total rows
  COUNT(*) - COUNT(DISTINCT order_id)                   AS duplicate_orders,  -- duplicate order_id
  COUNT_IF(customer_id IS NULL OR customer_id = '')     AS null_customer,     -- empty customer
  COUNT_IF(TRY_TO_TIMESTAMP(order_ts) IS NULL)          AS bad_order_ts,      -- timestamp doesn't parse
  COUNT_IF(TRY_CAST(quantity AS INT) IS NULL)           AS bad_quantity_type, -- quantity is not a number
  COUNT_IF(TRY_CAST(quantity AS INT) <= 0)              AS qty_le_0,          -- quantity <= 0
  COUNT_IF(TRY_CAST(unit_price AS DOUBLE) <= 0)         AS price_le_0         -- price <= 0
FROM workspace.shop.bronze_orders;                                            -- table from this same pipeline